# Protocol Comparison: Multi-Round SecureRAG with Ollama qwen3:0.6b

This notebook compares **Baseline**, **Diff Privacy**, and **Obfuscation** protocols for the same complicated question using local Ollama and multi-round retrieval.

## Goals

1. Run the same corpus and question across three protocols.
2. Keep model constant: local Ollama `qwen3:0.6b`.
3. Compare rounds used, context size, and answer style.
4. Inspect budget snapshots for protocol-specific behavior.

In [ ]:
from __future__ import annotations

import atexit
import socket
import subprocess
import sys
import time

import httpx

from securerag import PrivacyConfig, PrivacyProtocol, SecureRAGAgent
from securerag.corpus import CorpusBuilder
from securerag.llm import OllamaLLM
from securerag.models import RawDocument

In [ ]:
def free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return int(s.getsockname()[1])


def wait_for_health(base_url: str, timeout_s: float = 10.0) -> None:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            resp = httpx.get(f"{base_url}/health", timeout=0.5)
            if resp.status_code == 200:
                return
        except Exception:
            pass
        time.sleep(0.1)
    raise RuntimeError(f"sim_server did not become healthy at {base_url}")


port = free_port()
base_url = f"http://127.0.0.1:{port}"
sim_proc = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "securerag.sim_server:app",
        "--host",
        "127.0.0.1",
        "--port",
        str(port),
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
wait_for_health(base_url)
print("Backend ready at", base_url)


def cleanup() -> None:
    if sim_proc.poll() is None:
        sim_proc.terminate()


atexit.register(cleanup)

In [ ]:
documents = [
    RawDocument(doc_id="board", text="Board update: customer-facing reliability degraded in Q3 after delayed critical remediations."),
    RawDocument(doc_id="audit", text="Audit finding: privileged access reviews were overdue and owner assignment was inconsistent."),
    RawDocument(doc_id="vendor", text="Vendor analysis: concentration in two providers increased correlated outage exposure."),
    RawDocument(doc_id="ops", text="Operations telemetry: queue saturation and emergency changes preceded major timeout windows."),
    RawDocument(doc_id="policy", text="Policy requirement: critical controls need accountable owner, due date, and escalation path."),
    RawDocument(doc_id="risk", text="Risk committee: coupling between vendor concentration and remediation lag raises systemic risk."),
]

question = (
    "What is the most likely causal chain connecting policy drift, delayed remediation, and vendor concentration to Q3 reliability incidents, "
    "and what immediate and 90-day actions should leadership prioritize?"
)

llm = OllamaLLM(model="qwen3:0.6b", use_ollama=True)

In [ ]:
def run_protocol(protocol: PrivacyProtocol) -> dict:
    builder = (
        CorpusBuilder(protocol, backend_url=base_url)
        .with_chunk_size(170)
        .with_overlap(35)
        .add_documents(documents)
    )
    corpus = builder.build()

    cfg = PrivacyConfig(
        protocol=protocol,
        backend=base_url,
        epsilon=220.0,
        noise_std=0.22,
        k_decoys=3,
        top_k=4,
        max_rounds=6,
        paraphrase_decoys=True,
    )

    agent = SecureRAGAgent.from_config(cfg, corpus=corpus, llm=llm)
    result = agent.run(question)

    return {
        "protocol": protocol.name,
        "rounds": result.rounds,
        "context_size": result.context_size,
        "answer": result.answer,
        "budget": agent.budget_snapshot(),
    }


protocols = [
    PrivacyProtocol.BASELINE,
    PrivacyProtocol.DIFF_PRIVACY,
    PrivacyProtocol.OBFUSCATION,
]

runs = [run_protocol(p) for p in protocols]
runs

In [ ]:
for run in runs:
    print(f"=== {run['protocol']} ===")
    print("Rounds:", run["rounds"])
    print("Context size:", run["context_size"])
    print("Budget:", run["budget"])
    print("Answer preview:", run["answer"][:450], "...")
    print()

## Interpretation Notes

- **Baseline** usually gives direct retrieval behavior without noise or decoys.
- **Diff Privacy** introduces noisy embedding retrieval and budget accounting.
- **Obfuscation** adds decoys (optionally paraphrased) to hide intent patterns.

For fair comparison, keep the question, corpus, and LLM fixed as above.

In [ ]:
if sim_proc.poll() is None:
    sim_proc.terminate()
print("Stopped local sim backend")